# SFT v2: Qwen2.5-1.5B, LoRA va QLoRA (Kaggle GPU)

Qwen2.5-1.5B base modelini avval LoRA, keyin QLoRA bilan o'qitadi, 68 ta test savolida baholaydi va adapterlarni yuklab olish uchun zip qiladi.

Ikkala usul **bir xil sozlamalarda** (5 epoch, rank 64, eng yaxshi epoch saqlanadi), data esa v1 niki: farq faqat model hajmidan (0.5B -> 1.5B) va usuldan (LoRA / QLoRA) keladi.

Boshlashdan oldin o'ng tomondagi **Settings** bo'limida:

- **Accelerator -> GPU T4 x2** (yoki GPU P100)
- **Internet -> On** (buning uchun Kaggle akkaunti telefon raqami bilan tasdiqlangan bo'lishi kerak)

Keyin **Run All** ni bosing.

## 1. GPU borligini tekshirish

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Loyiha va base modelni yuklab olish

Kod va dataset GitHub'dan, Qwen2.5-1.5B (~3 GB) Hugging Face'dan olinadi.

In [ ]:
%cd /kaggle/working
!rm -rf Post_Training
!GIT_LFS_SKIP_SMUDGE=1 git clone -q https://github.com/davronbek-malikov/Post_Training.git
%cd Post_Training
!pip -q install -U transformers peft bitsandbytes accelerate
!pip -q uninstall -y torchao  # Kaggle'dagi eski torchao peft bilan adapter yuklashni buzadi

from huggingface_hub import snapshot_download
snapshot_download("Qwen/Qwen2.5-1.5B", local_dir="Qwen2.5-1.5B")
!ls -lh Qwen2.5-1.5B/*.safetensors
!wc -l sft/train.jsonl sft/val.jsonl

## 3. LoRA bilan o'qitish

Base model to'liq aniqlikda (32 bit, ~6 GB) yuklanadi. Xotiraga sig'ishi uchun gradient checkpointing yoqilgan.

Avval 3 qadamlik qisqa sinov: xotira yetadimi va bir qadam qancha vaqt olishini ko'rsatadi.

In [ ]:
!python sft_v2/train.py --method lora --max-steps 3

In [ ]:
!python sft_v2/train.py --method lora --epochs 5 --rank 64

## 4. LoRA adapterni zip qilish

Har bir o'qitishdan keyin darhol saqlanadi: sessiya uzilib qolsa ham adapter yo'qolmaydi.

In [ ]:
!rm -f /kaggle/working/sft_v2_adapters.zip
!zip -q -r /kaggle/working/sft_v2_adapters.zip chat_ui/adapters/sft-lora-1.5b
!ls -lh /kaggle/working/sft_v2_adapters.zip

## 5. QLoRA bilan o'qitish

Base model 4-bit ko'rinishda yuklanadi. Chiqishning boshida `QLoRA: ... ta qatlam 4-bit` degan qator bo'lishi kerak.

In [ ]:
!python sft_v2/train.py --method qlora --epochs 5 --rank 64

In [ ]:
!zip -q -r /kaggle/working/sft_v2_adapters.zip chat_ui/adapters/sft-qlora-1.5b
!ls -lh /kaggle/working/sft_v2_adapters.zip

## 6. Baholash: 68 ta test savoli

Avval o'qitilmagan Qwen2.5-1.5B (solishtirish uchun), keyin ikkala adapter.

0.5B natijalari: base 5/68, LoRA 32/68, QLoRA 32/68.

In [ ]:
!python eval/run_eval.py --base Qwen2.5-1.5B --name base-1.5b | tail -12
!python eval/run_eval.py --base Qwen2.5-1.5B --model sft-lora-1.5b | tail -12
!python eval/run_eval.py --base Qwen2.5-1.5B --model sft-qlora-1.5b | tail -12
!zip -q /kaggle/working/sft_v2_adapters.zip eval/results/base-1.5b.json eval/results/sft-lora-1.5b.json eval/results/sft-qlora-1.5b.json
!python eval/summarize.py

## 7. Yuklab olish

Quyidagi havolani bosing, yoki o'ng tomondagi **Output** bo'limidan `sft_v2_adapters.zip` ni yuklab oling.
Uni loyiha papkasida oching: adapterlar `chat_ui/adapters/` ga, natijalar `eval/results/` ga tushadi.

In [ ]:
%cd /kaggle/working
from IPython.display import FileLink
FileLink("sft_v2_adapters.zip")